# Notebook 20：力控制、阻抗控制与导纳控制

## 1. 本节在知识体系中的位置

```
NB19 操作空间控制 ──→ NB20 力/阻抗/导纳控制
NB17 关节控制 ──→ (交互控制 — 从运动到力)
```

工业机器人长期只做位置控制——"把手移动到 X"。但与人或环境交互时，纯位置控制会因刚度不匹配导致巨大接触力。阻抗控制让机器人表现为可调整的质量-弹簧-阻尼系统。

## 2. 学习目标

- ⭐ 理解自然约束与人工约束
- ⭐ 理解阻抗控制与导纳控制的原理和区别
- ⭐ 掌握"阻抗 vs 导纳选择准则"（按环境刚度）
- ⭐ 理解混合力/位置控制的选择矩阵 $\mathbf{S}$
- 📖 无源性与耦合稳定性概念

## 3. 阻抗控制 ⭐

### 3.1 核心思想 (Hogan, 1985)

阻抗控制不直接指定位置或力，而是指定**位置与力的动态关系**——即"阻抗"。目标阻抗：

$$\mathbf{M}_d(\ddot{\mathbf{x}} - \ddot{\mathbf{x}}_d) + \mathbf{D}_d(\dot{\mathbf{x}} - \dot{\mathbf{x}}_d) + \mathbf{K}_d(\mathbf{x} - \mathbf{x}_d) = \mathbf{F}_{ext}$$

物理直觉：当 $\mathbf{F}_{ext} = \mathbf{0}$（自由空间）时，机器人跟踪 $\mathbf{x}_d$。当与环境接触时（$\mathbf{F}_{ext} \neq \mathbf{0}$），机器人顺应外力偏离 $\mathbf{x}_d$。

- $\mathbf{M}_d$：虚拟质量——对外力的"惯性抵抗"
- $\mathbf{D}_d$：虚拟阻尼——对外力的"粘性抵抗"
- $\mathbf{K}_d$：虚拟刚度——对外力的"弹性抵抗"

### 3.2 控制律

将目标阻抗代入操作空间动力学：
$$\mathbf{F}_{imp} = \boldsymbol{\Lambda}\ddot{\mathbf{x}}_d + \boldsymbol{\mu}\dot{\mathbf{x}} + \mathbf{p} - \boldsymbol{\Lambda}\mathbf{M}_d^{-1}(\mathbf{D}_d\dot{\tilde{\mathbf{x}}} + \mathbf{K}_d\tilde{\mathbf{x}} - \mathbf{F}_{ext})$$

映射到关节：$\boldsymbol{\tau} = \mathbf{J}^T\mathbf{F}_{imp}$

## 4. 导纳控制 ⭐

### 4.1 与阻抗控制的区别

| | 阻抗控制 | 导纳控制 |
|---|---|---|
| 因果 | 测量运动 → 输出力 | 测量力 → 输出运动 |
| 外环 | — | 导纳模型修改期望运动 |
| 内环 | 力/力矩控制 | 高增益运动控制 |
| 适合 | 轻质、高反驱性机器人 | 重质、高减速比工业机器人 |

### 4.2 选择准则

- **软环境**（泡沫、人体）：阻抗控制——需要力控制精度
- **硬环境**（金属、混凝土）：导纳控制——力控不稳定，运动控稳定
- 本质原因：耦合稳定性取决于机器人与环境的刚度比

## 5. 混合力/位置控制

用选择矩阵 $\mathbf{S} = \text{diag}(s_1, \dots, s_6)$ 将任务空间分解为力控方向和位控方向：

$$\mathbf{F}_{hybrid} = \mathbf{S}\mathbf{F}_{force\_cmd} + (\mathbf{I} - \mathbf{S})\mathbf{F}_{motion\_cmd}$$

例如：拧螺丝——沿轴向控力（防止脱扣），绕轴向控位置（转动角度）。

## 6. Python 演示

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
%matplotlib inline
print("✅ 导入完成")

### 6.1 1D 阻抗控制 vs 导纳控制

In [ ]:
# 1D 仿真：质量块与环境刚性表面接触
dt = 0.001; T = 3.0; n_steps = int(T/dt)
x_wall = 1.0  # 墙壁位置
K_env = 500.0  # 环境刚度

# 阻抗参数
M_d, D_d, K_d = 1.0, 20.0, 100.0
m_robot = 5.0  # 机器人惯性

# 阻抗控制：控制力 → 产生运动
x_imp, x_dot_imp = 0.0, 0.0
x_imp_hist, f_imp_hist = [], []

# 导纳控制：测量力 → 修正期望运动
x_adm, x_dot_adm = 0.0, 0.0
x_d_adm = 1.2  # 初始期望位置（在墙内！）
x_adm_hist, f_adm_hist = [], []

x_des = 1.2  # 期望最终位置（穿透墙）

for i in range(n_steps):
    t = i * dt
    # 期望轨迹：从 0 到 1.2 的五次多项式（简化：斜坡）
    x_d_t = min(1.2, t * 1.0)

    # --- 阻抗控制 ---
    F_ext_imp = max(0, K_env * (x_imp - x_wall))  # 接触力
    # 阻抗律: M_d(ẍ - ẍ_d) + D_d(ẋ - ẋ_d) + K_d(x - x_d) = F_ext
    x_ref = x_d_t
    x_ddot_des = (K_d*(x_ref - x_imp) + D_d*(0 - x_dot_imp) + F_ext_imp) / M_d
    x_ddot_imp = 0 + x_ddot_des  # 简化
    x_dot_imp += x_ddot_imp * dt; x_imp += x_dot_imp * dt

    # --- 导纳控制 ---
    F_ext_adm = max(0, K_env * (x_adm - x_wall))
    # 导纳律: x_d 被修正 = F_ext / (M_d s² + D_d s + K_d) — 简化为准静态
    x_d_adm = x_d_t - F_ext_adm / K_d * 0.8
    x_ddot_adm = 200*(x_d_adm - x_adm) - 10*x_dot_adm  # 内环 PD（高增益）
    x_dot_adm += x_ddot_adm * dt; x_adm += x_dot_adm * dt

    x_imp_hist.append(x_imp); f_imp_hist.append(F_ext_imp)
    x_adm_hist.append(x_adm); f_adm_hist.append(F_ext_adm)

t_arr = np.linspace(0, T, n_steps)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].plot(t_arr, x_imp_hist, 'b-', linewidth=1.5, label='Actual Position')
axes[0,0].axhline(y=x_wall, color='r', linestyle='--', alpha=0.5, label='Wall (x=1.0)')
axes[0,0].set_ylabel('x (m)'); axes[0,0].set_title('Impedance Control — Position'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(t_arr, f_imp_hist, 'r-', linewidth=1.5)
axes[0,1].set_ylabel('F_ext (N)'); axes[0,1].set_title('Impedance Control — Contact Force'); axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(t_arr, x_adm_hist, 'b-', linewidth=1.5)
axes[1,0].axhline(y=x_wall, color='r', linestyle='--', alpha=0.5)
axes[1,0].set_xlabel('t (s)'); axes[1,0].set_ylabel('x (m)'); axes[1,0].set_title('Admittance Control — Position'); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(t_arr, f_adm_hist, 'r-', linewidth=1.5)
axes[1,1].set_xlabel('t (s)'); axes[1,1].set_ylabel('F_ext (N)'); axes[1,1].set_title('Admittance Control — Contact Force'); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/20_impedance_vs_admittance.png', dpi=100, bbox_inches='tight')
plt.show()

print("阻抗控制：机器人推墙壁，被弹回（弹簧-质量-阻尼行为）")
print("导纳控制：接触力使外环修正运动指令，内环精确跟踪修正后的轨迹")

### 6.2 不同刚度对接触行为的影响

In [ ]:
Kd_values = [20, 100, 500]
fig, ax = plt.subplots(figsize=(10, 5))
for Kd_v in Kd_values:
    x = 0.0; x_dot = 0.0; x_h = [0.0]; f_h = [0.0]
    for i in range(2000):
        F = max(0, K_env*(x - x_wall))
        x_ref = 1.2
        x_ddot = (Kd_v*(x_ref - x) + 20*(0 - x_dot) + F)/M_d
        x_dot += x_ddot*dt; x += x_dot*dt
        x_h.append(x); f_h.append(F)
    ax.plot(np.linspace(0, 2, 2001), x_h, linewidth=1.5, label=f'K_d={Kd_v}')
ax.axhline(y=x_wall, color='r', linestyle='--', alpha=0.5, label='Wall')
ax.set_xlabel('t (s)'); ax.set_ylabel('x (m)')
ax.set_title('Effect of Virtual Stiffness K_d on Contact Behavior')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/20_stiffness_effect.png', dpi=100, bbox_inches='tight')
plt.show()
print("K_d 越大 → 机器人越'硬' → 穿透越少 → 但接触力越大")

## 7. 练习题

### 概念题
1. 阻抗控制和导纳控制的本质区别是什么？什么时候选哪个？
2. 混合力/位置控制中的 $\mathbf{S}$ 矩阵怎么选？

### 编程题
1. 实现 2D 平面中的混合力/位置控制（沿 X 控力，沿 Y 控位置）。
2. 测量环境刚度变化时阻抗 vs 导纳的稳定性边界。

> 答案见 `solutions/20_solutions.ipynb`

## 8. 本节总结

| 方法 | 因果 | 适用环境 | 外环 | 内环 |
|------|------|----------|------|------|
| 阻抗 | 运动→力 | 软环境 | — | 力控 |
| 导纳 | 力→运动 | 硬环境 | 导纳模型 | 运动控（高增益） |
| 混合力/位 | 分开控制 | 约束明确 | S矩阵分解 | 力+运控并行 |